This notebook guides you through reproducing the ProToken MLSys paper results on Chameleon Cloud using the `chi` Python client. Run the cells in order in a CHI-enabled Jupyter environment where you have a valid lease and access to at least two NVIDIA A100 GPU (the PyTorch version used here does not support older GPU architectures).

In [ ]:
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

Replace `<name-of-your-lease>` with the name of an existing CHI lease that has suitable GPU nodes, then run this cell to bind the notebook to that lease.

In [ ]:
lease_name = "<name-of-your-lease>"
l = lease.get_lease(lease_name)
l.show()

Here we create a server on the reserved node using the CUDA image, attach a floating IP, and verify connectivity.

In [ ]:
username = os.getenv('USER')
s = server.Server(
    f"node-{lease_name}-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-CUDA"
)
s.submit(idempotent=True)

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")

This section runs commands on the provisioned node: update packages, install `uv`, clone the `protoken` repo, install dependencies (with the `chameleon` extra), and then execute `reproduce.sh` to run RQ1–RQ4 with the `smollm` model. Each RQ cell may take several minutes.

In [ ]:
s.execute("sudo apt update")

In [ ]:
s.execute("curl -LsSf https://astral.sh/uv/install.sh | sh")

In [ ]:
s.execute("git clone https://github.com/ahmayun/protoken.git")

In [ ]:
s.execute("cd protoken && ~/.local/bin/uv sync --extra chameleon")

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --rq1")

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --rq2")

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --rq3")

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --rq4")